# Prague flood exposure on Wherobots: maps

Renders the three maps with SedonaKepler (kepler.gl) from the committed `prague_flood`
package. Their colours, layers, tooltips and starting views are defined in `maps.py`, so
there is nothing to style by hand: run, take screenshots, destroy the runtime.

Needs the same files next to this notebook as `wherobots_run.ipynb` (the code zip and the
8 `.parquet` inputs), or a `RAW_URI` that already holds the inputs from that notebook.

**Screenshots:** collapse kepler's side panel with the `<` arrow at its top right, and open
the legend with the list icon in the column of buttons on the right. On Windows,
Win+Shift+S takes a screenshot of a region.

In [ ]:
# 1. Settings.
# The S3 folder the workers read the inputs from. On the Storage page, open
# data > customer-..., then click the copy icon on the prague-flood row, and add /raw.
RAW_URI = "s3://PASTE-YOUR-PATH/prague-flood/raw"

In [ ]:
# 2. The Sedona session. Wherobots provides Spark and Sedona: no jars or settings needed.
from sedona.spark import SedonaContext

sedona = SedonaContext.create(SedonaContext.builder().getOrCreate())

In [ ]:
# 3. What this runtime is. Record these numbers in the README.
import importlib.metadata as metadata
import sys

print("Python", sys.version.split()[0])
print("Spark ", sedona.version)
for package in ("apache-sedona", "geopandas", "pyarrow", "pandas", "shapely"):
    try:
        print(f"{package:<14}", metadata.version(package))
    except metadata.PackageNotFoundError:
        print(f"{package:<14} not installed")
sc = sedona.sparkContext
print("default parallelism (cores)", sc.defaultParallelism)
print("shuffle partitions         ", sedona.conf.get("spark.sql.shuffle.partitions"))

In [ ]:
# 4. Load our package from the zip next to this notebook. The pipeline has no Python UDFs:
#    all the work runs as Spark SQL and Sedona functions in the JVM, so only this notebook
#    needs the package, and the workers never do.
from pathlib import Path

zips = sorted(Path().glob("prague_flood-*.zip"))
assert len(zips) == 1, f"want one prague_flood-*.zip next to the notebook: {[z.name for z in zips]}"
sc.addPyFile(str(zips[0].resolve()))

# This import must come after addPyFile, which puts the zip on the import path.
from prague_flood import config, pipeline  # noqa: E402

print("code:", zips[0].name)

In [ ]:
# 5. Put the inputs where every worker can read them. The Spark workers cannot read My
#    Files, so each input found next to this notebook is copied into RAW_URI on S3.
#    (Uploading straight into Storage also works, but its web upload skips large files.)
fs = sc._jvm.org.apache.hadoop.fs
hadoop_conf = sc._jsc.hadoopConfiguration()
for name in config.SPARK_INPUTS:
    local = Path(name).resolve()
    if local.exists():
        target = fs.Path(f"{RAW_URI.rstrip('/')}/{name}")
        source = fs.Path(local.as_uri())
        target.getFileSystem(hadoop_conf).copyFromLocalFile(False, True, source, target)
        print(f"copied    {name}")
    else:
        print(f"not here  {name} (it must already be in RAW_URI)")

In [ ]:
# 6. Are the inputs on S3 byte-identical to the pinned ones? Spark reads each file on a
#    worker and hashes it there: the same SHA-256 check the download stage makes locally.
import hashlib

hashes = sc.binaryFiles(RAW_URI).mapValues(lambda data: hashlib.sha256(data).hexdigest())
actual = {uri.rsplit("/", 1)[-1]: digest for uri, digest in hashes.collect()}
for name, sha in config.SPARK_INPUTS.items():
    if actual.get(name) == sha:
        status = "ok"
    elif name not in actual:
        status = "MISSING"
    else:
        status = "DIFFERENT"
    print(f"{status:<9} {name}")
assert all(actual.get(n) == sha for n, sha in config.SPARK_INPUTS.items()), "inputs differ"

In [ ]:
# 7. kepler.gl's Python package. If the runtime lacks it, install it here. The map widget
#    may then need the browser tab reloaded (F5); the kernel and its data survive that.
import importlib.util
import subprocess

if importlib.util.find_spec("keplergl") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "keplergl==0.4.0"])
    print("installed keplergl 0.4.0: if a map below stays blank, reload the tab and rerun it")
print("keplergl", metadata.version("keplergl"))

In [ ]:
# 8. The analysis, as in the run notebook, then the three maps. Each map is also saved as
#    a standalone HTML file in outputs/maps, which opens in any browser.
import time

from prague_flood import maps

started = time.perf_counter()
results = pipeline.compute(sedona, RAW_URI)
specs = maps.map_specs(results.exposure, results.districts, results.zones)
maps_ = maps.build(specs)
saved = maps.save(maps_, specs, Path("outputs") / "maps")
print(f"maps ready in {time.perf_counter() - started:.1f} s")
for name, path in saved.items():
    print(f"{name:<14} {path}  {path.stat().st_size / 1e6:.1f} MB")

## Exposed building footprint by district, defences holding

Districts shaded by the footprint of buildings touching a flood zone. Hover over a district for its numbers.

In [ ]:
# 9. Shown as an interactive map; also saved in outputs/maps.
maps_["districts"]

## Buildings in the flood zones: the Vltava-Berounka confluence

Opens on Radotín, Zbraslav and Lipence, where exposure is highest. Buildings are coloured by the most frequent flood they touch; zoom out to see the whole city.

In [ ]:
# 10. Shown as an interactive map; also saved in outputs/maps.
maps_["flood_zones"]

## If the defences fail: residual risk in central Prague

Opens on Holešovice, Karlín and the Old Town: the 4,610 buildings behind the city's flood defences, with the Q100 extent for context.

In [ ]:
# 11. Shown as an interactive map; also saved in outputs/maps.
maps_["defences_fail"]

## Finish

1. Take the screenshots (collapse the side panel, open the legend first).
2. In the file browser, open `outputs/maps/` and download the three `.html` files: each
   is the interactive map, shareable on its own.
3. **Destroy the runtime** in the Wherobots console. Billing stops only when it is
   destroyed.